# APUSH FRQ Grader v4 — Colab training

This notebook fine-tunes a QLoRA adapter from the **reviewed, private-use v4 dataset**. It intentionally uses `train_chat_v4_judged_reviewed.jsonl`, not the earlier unreviewed or draft v4 artifacts. Model adapters, logs, and evaluation results are stored on Google Drive so a completed run survives a Colab disconnect.

The v4 dataset is private/noncommercial and may not be redistributed. Do not upload its rows or evaluation outputs to a public repository.

## 1. Confirm the GPU

Attach a GPU runtime before continuing. A T4 or better is suitable for the default 0.5B run.

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Attach a GPU runtime before continuing.'

CUDA available: True


## 2. Clone a reproducible repository revision

Set `SLM_GIT_REF` to a commit SHA for a reproducible run; `main` is a convenient default for experimentation.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/aryanjverma/slm.git'
REPO = Path('/content/slm')
GIT_REF = os.environ.get('SLM_GIT_REF', 'main')

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', GIT_REF], cwd=REPO, check=True)
if GIT_REF == 'main':
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO, check=True)
print('Repository:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
os.chdir(REPO)

Repository: aaa485f1e926bbce07a4f38d568627d1af64485e


## 3. Install training dependencies

This installs the project and its QLoRA/Unsloth training dependencies.

In [3]:
!pip install -q -e ".[train]" sentencepiece tiktoken

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for apush-frq-grader-slm (pyproject.toml) ... done


## 4. Configure Hugging Face access and persistent Drive output

A Hugging Face token is optional for the public Qwen checkpoints. Save the adapter and logs to Drive; the reviewed training data remains in the repository checkout.

In [4]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print('HF_TOKEN loaded from Colab secrets.')
except Exception as exc:
    print(f'No Colab secret loaded ({exc}); model downloads will be anonymous.')

from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/slm_v4')
MODEL_DIR = RUN_ROOT / 'apush-frq-grader-v4'
EVAL_DIR = RUN_ROOT / 'eval'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print('Run root:', RUN_ROOT)

No Colab secret loaded (Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.); model downloads will be anonymous.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run root: /content/drive/MyDrive/slm_v4


## 4.5. Optional private-bundle upload (normally skip)

Skip this section when the cloned repository already contains `train_chat_v4_judged_reviewed.jsonl`. Use the upload cell only as a fallback when step 5 explicitly reports that the training JSONL is missing. Do not commit or push the ZIP.

In a separate Windows PowerShell terminal, create the small ZIP with the five reviewed files (clear `.v4_private_bundle` first so old large files are not included). The final ZIP should be only a few megabytes.

Then run the single Python cell below, select `v4_private_artifacts.zip`, and rerun step 5.

In [ ]:
from google.colab import files
import shutil

PRIVATE_REPO_ROOT = RUN_ROOT / 'private_repo'
PRIVATE_REPO_ROOT.mkdir(parents=True, exist_ok=True)
if not (PRIVATE_REPO_ROOT / 'artifacts/data/v4/judging/private_use_manifest_v4.json').exists():
    print('Upload v4_private_artifacts.zip created by the PowerShell command in the markdown above.')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
    assert zip_name, 'Select the .zip bundle created by Compress-Archive.'
    shutil.unpack_archive(zip_name, PRIVATE_REPO_ROOT, format='zip')
    print('Extracted private artifacts to', PRIVATE_REPO_ROOT)
else:
    print('Private v4 artifact bundle already present; no upload needed.')

: 

## 5. Lock the reviewed v4 training artifact

The reviewed v4 files are private and are not included in a fresh Git clone. Before running this section, copy the repository `artifacts` folder to Drive at `MyDrive/slm_v4/private_repo/artifacts` (or otherwise place the private bundle where the candidate path below can find it). This checks the private-use manifest hashes, the 226-row reviewed artifact, its audit status, and the v4 system prompt. It refuses to train if the reviewed artifact has been changed or replaced.

In [5]:
import hashlib
import json
from pathlib import Path

PRIVATE_REPO_ROOT = RUN_ROOT / 'private_repo'
candidate_roots = [REPO, PRIVATE_REPO_ROOT, RUN_ROOT]
manifest_rel = Path('artifacts/data/v4/judging/private_use_manifest_v4.json')
def candidate_is_usable(root):
    manifest_path = root / manifest_rel
    if not manifest_path.exists():
        return False
    candidate_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    train_record = next(item for item in candidate_manifest['artifacts'] if Path(item['path']).name == 'train_chat_v4_judged_reviewed.jsonl')
    train_path = root / train_record['path']
    audit_path = root / 'artifacts/data/v4/judging/training_audit_v4_reviewed.json'
    return (
        train_path.exists()
        and hashlib.sha256(train_path.read_bytes()).hexdigest() == train_record['sha256']
        and audit_path.exists()
    )
DATA_ROOT = next((root for root in candidate_roots if candidate_is_usable(root)), None)
if DATA_ROOT is not None:
    V4_ROOT = DATA_ROOT / 'artifacts/data/v4/judging'
else:
    # Also accept a Drive upload containing the judging directory itself.
    direct_roots = [RUN_ROOT / 'judging', RUN_ROOT / 'v4' / 'judging']
    V4_ROOT = next((root for root in direct_roots if (root / 'private_use_manifest_v4.json').exists()), None)
    DATA_ROOT = V4_ROOT
if V4_ROOT is None:
    raise FileNotFoundError(
        'Reviewed v4 artifacts are not in the Git checkout. Copy the private artifacts folder to '
        f'{PRIVATE_REPO_ROOT / "artifacts"}'
    )
CASE_DATA = V4_ROOT / 'train_cases_v4_judged_reviewed.jsonl'
TRAIN_DATA = V4_ROOT / 'train_chat_v4_judged_reviewed.jsonl'
AUDIT_PATH = V4_ROOT / 'training_audit_v4_reviewed.json'
PRIVATE_MANIFEST = V4_ROOT / 'private_use_manifest_v4.json'

assert CASE_DATA.exists() and TRAIN_DATA.exists() and AUDIT_PATH.exists() and PRIVATE_MANIFEST.exists(), 'Reviewed v4 artifacts are missing.'
manifest = json.loads(PRIVATE_MANIFEST.read_text(encoding='utf-8'))
assert manifest['usage_scope'] == 'private_noncommercial_assignment_no_redistribution'
assert manifest['redistribution_authorized'] is False
assert manifest['technical_audit_clean'] is True
assert manifest['case_count'] == 226

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

case_record = next(item for item in manifest['artifacts'] if Path(item['path']).name == CASE_DATA.name)
train_record = next(item for item in manifest['artifacts'] if Path(item['path']).name == TRAIN_DATA.name)
assert sha256(CASE_DATA) == case_record['sha256'], f'Reviewed case hash mismatch: {CASE_DATA}'
assert sha256(TRAIN_DATA) == train_record['sha256'], f'Training data hash mismatch: {TRAIN_DATA}'

audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
assert audit['total'] == 226
assert not audit['global_reasons'], audit['global_reasons']
assert audit['human_review_rate'] >= 0.10
print(f"Reviewed rows: {audit['total']}; human review rate: {audit['human_review_rate']:.1%}")

Reviewed rows: 226; human review rate: 10.2%


## 6. Validate chat format and token budget

The QLoRA script consumes the three-message chat rows directly. This preflight confirms that all rows use the full v4 rubric prompt, have a JSON assistant target, and fit the selected context window.

In [6]:
# Make imports work even if the install cell was run before the repo clone or after a cwd change.
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))
from transformers import AutoTokenizer
from apush_frq_grader_slm.dataset_v4 import v4_chat_row
from apush_frq_grader_slm.io import read_jsonl, write_jsonl
from apush_frq_grader_slm.prompts_v4 import V4_TRAIN_SYSTEM_PROMPT
from apush_frq_grader_slm.schemas import FRQCase
from apush_frq_grader_slm.training_v3 import assert_assistant_only_example, tokenize_assistant_only

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SEQ_LENGTH = 3072
repo_case_data = REPO / 'artifacts/data/v4/judging/train_cases_v4_judged_reviewed.jsonl'
drive_case_data = RUN_ROOT / 'private_repo/artifacts/data/v4/judging/train_cases_v4_judged_reviewed.jsonl'
CASE_DATA = repo_case_data if repo_case_data.exists() else drive_case_data
assert CASE_DATA.exists(), f'Reviewed v4 cases are missing: {CASE_DATA}'
reviewed_cases = [FRQCase.model_validate(row) for row in read_jsonl(CASE_DATA)]
rows = [v4_chat_row(case) for case in reviewed_cases]
TRAIN_DATA = RUN_ROOT / 'train_chat_v4_current_prompt.jsonl'
write_jsonl(TRAIN_DATA, rows)
assert len(rows) == 226
for index, row in enumerate(rows):
    messages = row['messages']
    assert [message['role'] for message in messages] == ['system', 'user', 'assistant'], index
    assert messages[0]['content'] == V4_TRAIN_SYSTEM_PROMPT, index
    target = json.loads(messages[-1]['content'])
    assert set(target) == {'scores', 'total', 'feedback'}, index
    assert target['total'] == sum(target['scores'].values()), index

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenized_rows = [tokenize_assistant_only(row['messages'], tokenizer, max_length=MAX_SEQ_LENGTH) for row in rows]
for example in tokenized_rows:
    assert_assistant_only_example(example)
lengths = [len(example['input_ids']) for example in tokenized_rows]
supervised_lengths = [sum(label != -100 for label in example['labels']) for example in tokenized_rows]
assert max(lengths) <= MAX_SEQ_LENGTH, (max(lengths), MAX_SEQ_LENGTH)
print(f'Rows={len(rows)}; total tokens min/median/max={min(lengths)}/{sorted(lengths)[len(lengths)//2]}/{max(lengths)}')
print(f'Assistant labels min/median/max={min(supervised_lengths)}/{sorted(supervised_lengths)[len(supervised_lengths)//2]}/{max(supervised_lengths)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Rows=226; total tokens min/median/max=970/1166/1706
Assistant labels min/median/max=114/171/436


## 7. Configure the v4 QLoRA run

The default uses three epochs (about 45 optimizer steps) for 226 examples at effective batch size 16, with a 5% warmup. `RUN_ID` creates a fresh versioned adapter directory so an older v4 adapter cannot be reused accidentally. Increment it for another fresh run.

In [7]:
import math
import torch

RUN_TRAINING = True
RUN_ID = 'assistant-only-r1'  # Increment this for every intentionally fresh run.
MODEL_DIR = RUN_ROOT / f'apush-frq-grader-v4-{RUN_ID}'
FINAL_DIR = MODEL_DIR / 'final'
BATCH_SIZE = 1
GRAD_ACCUM = 16
EPOCHS = 3
WARMUP_RATIO = 0.05
LORA_RANK = 16
LEARNING_RATE = 2e-4
SEED = 13
ESTIMATED_STEPS = math.ceil(len(rows) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
ADAPTER_CONFIG = FINAL_DIR / 'adapter_config.json'
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f'Fresh v4 run={RUN_ID}: {len(rows)} rows, ~{ESTIMATED_STEPS} steps, warmup={WARMUP_RATIO:.0%}, output={MODEL_DIR}')

Fresh v4 run=assistant-only-r1: 226 rows, ~45 steps, warmup=5%, output=/content/drive/MyDrive/slm_v4/apush-frq-grader-v4-assistant-only-r1


## 8. Train and save the v4 adapter

`train_v4.py` applies loss only to assistant tokens, logs every optimizer step visibly, saves an epoch checkpoint, and writes the final adapter under the versioned Drive directory. Rerunning resumes the latest complete checkpoint after a disconnect.

In [8]:
TRAIN_SCRIPT = REPO / 'scripts/train_v4.py'
assert TRAIN_SCRIPT.exists(), (
    f'Missing {TRAIN_SCRIPT}. Commit and push scripts/train_v4.py, then rerun the clone/install cells.'
)
if RUN_TRAINING and not ADAPTER_CONFIG.exists():
    command = [
        sys.executable, str(TRAIN_SCRIPT),
        '--model', BASE_MODEL,
        '--data', str(TRAIN_DATA),
        '--output', str(MODEL_DIR),
        '--max-seq-length', str(MAX_SEQ_LENGTH),
        '--lora-rank', str(LORA_RANK),
        '--batch-size', str(BATCH_SIZE),
        '--grad-accum', str(GRAD_ACCUM),
        '--epochs', str(EPOCHS),
        '--warmup-ratio', str(WARMUP_RATIO),
        '--learning-rate', str(LEARNING_RATE),
        '--seed', str(SEED),
    ]
    if USE_BF16:
        command.append('--bf16')
    checkpoints = sorted(MODEL_DIR.glob('checkpoint-*'), key=lambda path: int(path.name.split('-')[-1]))
    complete_checkpoints = [path for path in checkpoints if (path / 'trainer_state.json').exists()]
    if complete_checkpoints:
        command.extend(['--resume-from-checkpoint', str(complete_checkpoints[-1])])
        print('Resuming:', complete_checkpoints[-1])
    log_path = RUN_ROOT / f'v4_training_{RUN_ID}.log'
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    assert returncode == 0, f'Training failed; inspect {log_path}'
elif ADAPTER_CONFIG.exists():
    print(f'Adapter already complete: {MODEL_DIR}')
else:
    print('RUN_TRAINING=False; preflight completed without training.')

assert ADAPTER_CONFIG.exists() or not RUN_TRAINING, 'No saved adapter found after training.'

/usr/bin/python3: can't open file '/content/slm/scripts/train_v4.py': [Errno 2] No such file or directory


AssertionError: Training failed; inspect /content/drive/MyDrive/slm_v4/v4_training_assistant-only-r1.log

## 9. Smoke-test the saved adapter on a held-out evaluation case

This uses the v4 prompt and checks only output structure. It does not expose or train on any evaluation text. The full evaluation cell is opt-in because it can take substantial GPU time.

In [13]:
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    from apush_frq_grader_slm.dataset_v4 import format_v4_user_message
    from apush_frq_grader_slm.io import read_jsonl
    from apush_frq_grader_slm.schemas import FRQCase

    assert ADAPTER_CONFIG.exists(), 'Train the adapter before smoke testing.'
    eval_case = FRQCase.model_validate(next(iter(read_jsonl(REPO / 'artifacts/data/eval_cb_cases.jsonl'))))
    smoke_tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='auto')
    smoke_model = PeftModel.from_pretrained(base, FINAL_DIR).merge_and_unload().eval()
    messages = [
        {'role': 'system', 'content': V4_TRAIN_SYSTEM_PROMPT},
        {'role': 'user', 'content': format_v4_user_message(eval_case)},
    ]
    inputs = smoke_tokenizer(smoke_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True), return_tensors='pt').to(smoke_model.device)
    with torch.no_grad():
        output = smoke_model.generate(**inputs, max_new_tokens=320, do_sample=False, pad_token_id=smoke_tokenizer.eos_token_id)
    response = smoke_tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    parsed = json.loads(response)
    assert parsed['total'] == sum(parsed['scores'].values())
    print(json.dumps(parsed, indent=2))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "scores": {
    "thesis": 1,
    "contextualization": 0,
    "evidence": 1,
    "analysis_reasoning": 2
  },
  "total": 4,
  "feedback": {
    "thesis": "The claim that transatlantic trade significantly changed British North American society remains broad and does not explicitly establish a causal relationship.",
    "contextualization": "The passage about Massachusetts Bay does not set a broader historical context beyond the immediate period being evaluated.",
    "evidence": "The Massachussetts Bay Colony\u2019s religious tolerance and shipbuilding industry are used to explain regional differentiation, but they do not sustain a causative link between trade and social change.",
    "analysis_reasoning": "The essay uses comparative reasoning to explain regional differences while acknowledging continuity through slavery."
  }
}


## 10. Optional held-out evaluation

This evaluates against the College Board-derived held-out set with the exact v4 full-rubric prompt and saves resumable per-case results to a versioned Drive directory. It is enabled by default so a training run is not considered complete without evaluation.

In [16]:
RUN_EVALUATION = True
MODEL_NAME = f'apush-frq-grader-v4-{RUN_ID}'
RUN_EVAL_DIR = EVAL_DIR / RUN_ID
if RUN_EVALUATION:
    subprocess.run([
        sys.executable, 'scripts/eval_hf_model.py',
        '--model', str(FINAL_DIR),
        '--model-name', MODEL_NAME,
        '--eval-path', 'artifacts/data/eval_cb_cases.jsonl',
        '--real-eval',
        '--prompt-version', 'v4',
        '--output-dir', str(RUN_EVAL_DIR),
        '--resume',
    ], check=True)
    print('Evaluation outputs:', sorted(path.name for path in RUN_EVAL_DIR.iterdir()))
    training_metadata = FINAL_DIR / 'v4_training_metadata.json'
    assert training_metadata.exists(), f'Missing fresh-training proof: {training_metadata}'
    print('Training metadata:')
    print(training_metadata.read_text(encoding='utf-8'))
    summary_path = RUN_EVAL_DIR / f'{MODEL_NAME}_real_summary.jsonl'
    assert summary_path.exists(), f'Missing evaluation summary: {summary_path}'
    print('Evaluation summary:')
    print(summary_path.read_text(encoding='utf-8'))

Evaluation outputs: ['apush-frq-grader-v4-assistant-only-r1_real_by_rubric.jsonl', 'apush-frq-grader-v4-assistant-only-r1_real_results.jsonl', 'apush-frq-grader-v4-assistant-only-r1_real_results_diagnostics.json', 'apush-frq-grader-v4-assistant-only-r1_real_results_dimensions.json', 'apush-frq-grader-v4-assistant-only-r1_real_summary.jsonl']
Training metadata:
{
  "assistant_only_loss": true,
  "epochs": 3.0,
  "learning_rate": 0.0002,
  "rows": 226,
  "seed": 13,
  "supervised_tokens": 41985,
  "warmup_ratio": 0.05
}

Evaluation summary:
{"model_name": "apush-frq-grader-v4-assistant-only-r1", "count": 53, "structured_output_valid_rate": 0.9811, "exact_match_rate": 0.3538, "within_one_rate": 0.8585, "total_exact_match_rate": 0.0, "total_within_one_rate": 0.1698, "qwk": 0.1158, "total_mae": 3.0755, "criterion_exact_rates": {"thesis": 0.5283, "contextualization": 0.3962, "evidence": 0.1698, "analysis_reasoning": 0.3208}, "rubric_accuracy_mean": 0.8585, "evidence_grounding_rate": 0.8868, 

## 11. Record the completed run

This writes a compact, non-sensitive run receipt next to the adapter. Do not add private training rows or per-case evaluation outputs to public version control.

In [15]:
if ADAPTER_CONFIG.exists():
    receipt = {
        'model_name': MODEL_NAME,
        'base_model': BASE_MODEL,
        'training_data': str(TRAIN_DATA),
        'training_data_sha256': sha256(TRAIN_DATA),
        'rows': len(rows),
        'max_seq_length': MAX_SEQ_LENGTH,
        'run_id': RUN_ID,
        'assistant_only_loss': True,
        'epochs': EPOCHS,
        'estimated_steps': ESTIMATED_STEPS,
        'warmup_ratio': WARMUP_RATIO,
        'lora_rank': LORA_RANK,
        'learning_rate': LEARNING_RATE,
        'seed': SEED,
        'repository_revision': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(),
        'usage_scope': manifest['usage_scope'],
    }
    receipt_path = RUN_ROOT / 'v4_run_receipt.json'
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    print(receipt_path)
    print(json.dumps(receipt, indent=2))

/content/drive/MyDrive/slm_v4/v4_run_receipt.json
{
  "model_name": "apush-frq-grader-v4-assistant-only-r1",
  "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
  "training_data": "/content/drive/MyDrive/slm_v4/train_chat_v4_current_prompt.jsonl",
  "training_data_sha256": "0e612c43ffd31620dcf5ac146e7f579259e8154da9eb5e3bd8758d9077bf1720",
  "rows": 226,
  "max_seq_length": 3072,
  "run_id": "assistant-only-r1",
  "assistant_only_loss": true,
  "epochs": 3,
  "estimated_steps": 45,
  "warmup_ratio": 0.05,
  "lora_rank": 16,
  "learning_rate": 0.0002,
  "seed": 13,
  "repository_revision": "3caaed20b44fe3f33b7e6466b9a111557348686a",
  "usage_scope": "private_noncommercial_assignment_no_redistribution"
}


## 12. Export the aggregate v4 report

This creates a shareable aggregate-only report bundle on Drive. It deliberately excludes per-case results and held-out essays.

In [ ]:
import shutil
import zipfile

REPORT_DIR = RUN_ROOT / 'reports' / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = RUN_EVAL_DIR / f'{MODEL_NAME}_real_summary.jsonl'
summary = json.loads(summary_path.read_text(encoding='utf-8').splitlines()[0])
report_text = f'''# APUSH FRQ Grader v4 Run Report

Model: `{MODEL_NAME}`  
Base: `{BASE_MODEL}`  
Training: {len(rows)} reviewed rows, assistant-only QLoRA, {EPOCHS} epochs, {WARMUP_RATIO:.0%} warmup  
Evaluation: {summary['count']} held-out College Board-derived cases

| Metric | Result |
|---|---:|
| Structured-output validity | {summary['structured_output_valid_rate']:.2%} |
| Criterion exact match | {summary['exact_match_rate']:.2%} |
| Total-score exact match | {summary['total_exact_match_rate']:.2%} |
| Total within one point | {summary['total_within_one_rate']:.2%} |
| Total-score MAE | {summary['total_mae']:.4f} |
| Total-score QWK | {summary['qwk']:.4f} |
| Evidence grounding | {summary['evidence_grounding_rate']:.2%} |
| Mean predicted total | {summary['total_score_mean']:.3f} |

Conclusion: output validity and grounding are strong, but score calibration is not acceptable. This run severely under-scores essays and is not production-ready.

Privacy: share aggregate files only; do not redistribute training rows, held-out essays, or per-case results.
'''
report_path = REPORT_DIR / 'v4_run_report.md'
report_path.write_text(report_text, encoding='utf-8')
aggregate_files = [
    summary_path,
    RUN_EVAL_DIR / f'{MODEL_NAME}_real_by_rubric.jsonl',
    RUN_EVAL_DIR / f'{MODEL_NAME}_real_results_dimensions.json',
    RUN_EVAL_DIR / f'{MODEL_NAME}_real_results_diagnostics.json',
    FINAL_DIR / 'v4_training_metadata.json',
    receipt_path,
]
for path in aggregate_files:
    if path.exists():
        shutil.copy2(path, REPORT_DIR / path.name)
bundle_path = RUN_ROOT / f'v4_report_{RUN_ID}.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in REPORT_DIR.iterdir():
        bundle.write(path, arcname=path.name)
print('Report:', report_path)
print('Download bundle:', bundle_path)
print(report_text)

DOWNLOAD_REPORT_BUNDLE = False
if DOWNLOAD_REPORT_BUNDLE:
    from google.colab import files as colab_files
    colab_files.download(str(bundle_path))